**PRÁCTICA 5 REACCIONES DE PARTIDOS POLÍTICOS SEGÚN TU EMOCIÓN**

**Escenario de aplicación.** La persona se pone delante de la cámara, elige un partido político con las teclas y, según la expresión que ponga (enfado, miedo, alegría, neutral, tristeza, sorpresa o *duckface*), aparece encima de su cabeza una foto que hace referencia a dicho partido y emoción.

**Qué información del rostro se usa:**
1. **Detector** DNN de OpenCV para la caja de la cara.
2. **Landmarks** de MediaPipe Face Landmarker →
   - *alineación*: se gira la imagen para dejar los ojos en horizontal antes de clasificar.
   - *colocación*: la caricatura se ancla al punto de la frente y se gira con la inclinación de la cabeza.
3. **Modelo biométrico propio**: CNN tipo mini-Xception entrenada por nosotros sobre FER-2013 + duckface.

Con la tecla `m` se puede cambiar en directo entre la **CNN propia**, los **modelos clásicos** (uno tras otro: HOG+SVM, LBP+SVM, HOG+LBP+PCA+SVM, autocaras+SVM, HOG+PCA+KNN) y **DeepFace**, mostrando el funcionamiento de cada uno.

**Teclas:** `n`/`b` partido siguiente/anterior · `m` cambiar modelo · `l` mostrar landmarks · `f` pantalla completa · `Esc` salir.

In [13]:
import os, threading, time
import cv2
import joblib
import numpy as np

from comun import (Camara, DetectorCaras, Landmarks, alinear, cargar_imagenes,
                   recortar_cara, sticker, texto, transformar_caja)
from comun import EMOCIONES, TAM_CNN, preparar_cnn
from selector import SelectorImagenes

**CONFIGURACIÓN**

- `RUTA_CNN` / `RUTAS_CLASICO`: modelos propios. Cada clásico de la lista es un modo distinto de la tecla `m`; se omiten los que no existan o no tengan las 7 clases actuales (dataset antiguo).
- `ALPHA_SUAVIZADO`: peso del frame nuevo en la media móvil exponencial de probabilidades.
- `ESCALA_IMG`: ancho de la caricatura respecto al ancho de la cara.

In [14]:
RUTA_CNN = "modelos/cnn.keras"
RUTAS_CLASICO = [
    "modelos/hog_svm.joblib",
    "modelos/lbp_svm.joblib",
    "modelos/hoglbp_pca_svm.joblib",
    "modelos/autocaras_svm.joblib",
    "modelos/hog_pca_knn.joblib",
]

PARTIDOS = [
    {"nombre": "PSOE", "imagen": "logos-partidos/psoe.png", "etiqueta": "psoe"},
    {"nombre": "PP", "imagen": "logos-partidos/pp.png", "etiqueta": "pp"},
    {"nombre": "Vox", "imagen": "logos-partidos/vox.png", "etiqueta": "vox"},
    {"nombre": "Frente-Amplio", "imagen": "logos-partidos/frente-amplio.png", "etiqueta": "frente-amplio"},
]
CARPETA_REACCIONES = "emotion-images"

VENTANA = "Emociones politicas"
ALPHA_SUAVIZADO = 0.35
ESCALA_IMG = 1.6
USAR_ALINEACION = True

**MODELOS DE EMOCIÓN**

Los tres modelos comparten la misma interfaz `probs(cara_gris) -> vector de 7 probabilidades`:

- **`ModeloCNN`**: carga `cnn.keras` y envuelve la llamada en `tf.function`.
- **`ModeloClasico(ruta)`**: pipeline de scikit-learn guardado con su extractor; se crea una instancia por cada archivo de `RUTAS_CLASICO`. La SVM no da probabilidades, así que se aplica un softmax a su `decision_function`.
- **`ModeloDeepFace`**: `DeepFace.analyze(actions=["emotion"])` sin volver a detectar la cara. DeepFace tiene `disgust` se suma a `angry` y no tiene `duckface`.

In [15]:
N = len(EMOCIONES)

def softmax(z):
    z = np.asarray(z, np.float64) - np.max(z)
    e = np.exp(z)
    return e / e.sum()


class ModeloCNN:
    nombre = "CNN propia"

    def __init__(self, ruta=RUTA_CNN):
        import keras
        import tensorflow as tf
        self.tf = tf
        modelo = keras.models.load_model(ruta)
        self._f = tf.function(lambda x: modelo(x, training=False))
        self._f(tf.zeros((1, TAM_CNN, TAM_CNN, 1)))              # trazado del grafo

    def probs(self, cara):
        x = self.tf.constant(preparar_cnn(cara)[None])
        return self._f(x).numpy()[0]


class ModeloClasico:
    def __init__(self, ruta):
        m = joblib.load(ruta)
        if len(m.classes_) != N:
            raise ValueError(f"{ruta} tiene {len(m.classes_)} clases (dataset antiguo): reentrénalo")
        self.modelo = m
        self.nombre = f"Clasico ({os.path.splitext(os.path.basename(ruta))[0]})"

    def probs(self, cara):
        p = np.zeros(N)
        if hasattr(self.modelo, "predict_proba"):
            p[self.modelo.classes_] = self.modelo.predict_proba([cara])[0]
        else:
            p[self.modelo.classes_] = softmax(self.modelo.decision_function([cara])[0])
        return p


class ModeloDeepFace:
    nombre = "DeepFace"
    MAPA = {"angry": "angry", "disgust": "angry", "fear": "fear", "happy": "happy",
            "sad": "sad", "surprise": "surprise", "neutral": "neutral"}

    def __init__(self):
        from deepface import DeepFace
        self.df = DeepFace
        self.ultimo, self.ocupado, self.ms = None, False, 0.0
        self._analizar(np.zeros((48, 48), np.uint8))              # carga del modelo

    def _analizar(self, cara):
        r = self.df.analyze(cv2.cvtColor(cara, cv2.COLOR_GRAY2BGR), actions=["emotion"],
                            detector_backend="skip", enforce_detection=False, silent=True)[0]["emotion"]
        p = np.zeros(N)
        for k, v in r.items():
            p[EMOCIONES.index(self.MAPA[k])] += float(v) / 100
        return p

    def _trabajo(self, cara):
        t0 = time.time()
        try:
            self.ultimo = self._analizar(cara)
        except Exception as e:
            print("DeepFace:", e)
        finally:
            self.ms, self.ocupado = 1000 * (time.time() - t0), False

    def probs(self, cara):
        if not self.ocupado:
            self.ocupado = True
            threading.Thread(target=self._trabajo, args=(cara.copy(),), daemon=True).start()
        return self.ultimo


# Un modo por modelo: CNN, cada clásico disponible y DeepFace
MODOS = ([("CNN", ModeloCNN)]
         + [(os.path.basename(r), (lambda r=r: ModeloClasico(r))) for r in RUTAS_CLASICO if os.path.exists(r)]
         + [("DeepFace", ModeloDeepFace)])
_cargados, _fallidos = {}, set()


def obtener_modelo(i):
    i %= len(MODOS)
    if i in _fallidos:
        raise RuntimeError(f"{MODOS[i][0]} no disponible")
    if i not in _cargados:
        print("Cargando", MODOS[i][0], "...")
        try:
            _cargados[i] = MODOS[i][1]()
        except Exception:
            _fallidos.add(i)
            raise
    return _cargados[i]


**ESTABILIZACIÓN TEMPORAL**

En lugar de votar la etiqueta de los últimos N frames, se hace una **media móvil exponencial de las probabilidades**. Así la confianza mostrada es la probabilidad real (suavizada) y no un recuento de votos, la caricatura no parpadea cuando el modelo duda entre dos emociones y la reacción tras un cambio de expresión es más rápida que con una ventana fija. La caja de la cara se suaviza igual para que no tiemble.

In [16]:
class Suavizador:
    def __init__(self, alpha=ALPHA_SUAVIZADO):
        self.alpha, self.p = alpha, None

    def actualizar(self, p):
        if p is not None:
            p = np.asarray(p, np.float64)
            self.p = p.copy() if self.p is None else (1 - self.alpha) * self.p + self.alpha * p
        return self.p

    def reset(self):
        self.p = None


def suavizar_caja(anterior, nueva, alpha=0.5):
    if anterior is None:
        return nueva
    return tuple(int(alpha * n + (1 - alpha) * a) for a, n in zip(anterior, nueva))

**PROCESADO DE UN FRAME**

`procesar_frame` hace todo el trabajo de un fotograma y se puede probar con una imagen fija:
1. Detecta la cara más grande con el DNN.
2. Obtiene los landmarks; si los hay, gira la imagen en gris para poner los ojos en horizontal y lleva la caja a la imagen girada.
3. Recorta la cara con `recortar_cara` y obtiene las probabilidades del modelo activo.
4. Suaviza las probabilidades y dibuja la caja, la emoción y la caricatura del partido anclada a la frente y girada con la cabeza.

In [ ]:
detector = DetectorCaras(conf_min=0.5)
landmarks = Landmarks()
selector = SelectorImagenes(PARTIDOS, lado="derecha", ancho_sel=80)


def reacciones_partido():
    return cargar_imagenes(os.path.join(CARPETA_REACCIONES, selector.nombre.lower()))


def colocar_caricatura(frame, img, caja, pts):
    x, y, w, h = caja
    ih, iw = img.shape[:2]
    ancho = w * ESCALA_IMG
    if pts is not None:
        ang = np.radians(Landmarks.angulo(pts))
        arriba = np.array([np.sin(ang), -np.cos(ang)])          
        base = pts[Landmarks.FRENTE] + arriba * 0.12 * h
        giro = -np.degrees(ang)
    else:
        arriba, base, giro = np.array([0.0, -1.0]), np.array([x + w / 2, y - 5.0]), 0.0
    alto = ancho * ih / iw
    hueco = max(base[1], 0.45 * h)                              
    if alto > hueco:
        ancho, alto = ancho * hueco / alto, hueco
    centro = base + arriba * alto / 2
    sticker(frame, img, centro, ancho, giro)


def dibujar_landmarks(frame, pts):
    for p in pts[::4]:
        cv2.circle(frame, (int(p[0]), int(p[1])), 1, (0, 255, 255), -1)
    a, b = Landmarks.ojos(pts)
    cv2.line(frame, tuple(a.astype(int)), tuple(b.astype(int)), (255, 0, 255), 2)
    cv2.circle(frame, tuple(pts[Landmarks.FRENTE].astype(int)), 4, (255, 0, 255), -1)


def procesar_frame(frame, modelo, suav, imagenes, estado, mostrar_lm=False):
    # Dibuja sobre `frame` y devuelve emocion, confianza, ms_modelo o None si no hay cara
    gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    nueva = detector.mayor(frame)
    if nueva is None:
        estado["caja"] = None
        suav.reset()
        return None
    caja = estado["caja"] = suavizar_caja(estado.get("caja"), nueva)

    pts = landmarks.detectar(frame)
    if pts is not None and USAR_ALINEACION:
        gris_al, M = alinear(gris, pts)
        caja_al = transformar_caja(caja, M)
    else:
        gris_al, caja_al = gris, caja

    cara = recortar_cara(gris_al, caja_al)
    if cara is None:
        return None
    t0 = time.perf_counter()
    p = suav.actualizar(modelo.probs(cara))
    ms = getattr(modelo, "ms", None) or 1000 * (time.perf_counter() - t0)

    x, y, w, h = caja
    if mostrar_lm and pts is not None:
        dibujar_landmarks(frame, pts)
    if p is None:                                               
        texto(frame, "analizando...", x, y + h + 25, 0.7)
        return None
    k = int(np.argmax(p))
    emocion, conf = EMOCIONES[k], float(p[k])
    if emocion in imagenes:
        colocar_caricatura(frame, imagenes[emocion], caja, pts)
    return emocion, conf, ms

print("Landmarks activos:", landmarks.ok)

**BUCLE PRINCIPAL CON WEBCAM**

La cámara se lee en un hilo para que el bucle siempre procese el último fotograma. En pantalla se muestran el modelo activo y su latencia por cara

In [ ]:
modo = 0
modelo = obtener_modelo(modo)
suav = Suavizador()
imagenes = reacciones_partido()
estado, mostrar_lm, full = {}, False, True
last_id = -1

cam = Camara()
cv2.namedWindow(VENTANA, cv2.WINDOW_NORMAL)
cv2.setWindowProperty(VENTANA, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
try:
    while True:
        fid, frame = cam.read()
        if frame is None or fid == last_id:
            if cv2.waitKey(1) & 0xFF == 27:
                break
            continue
        last_id = fid

        res = procesar_frame(frame, modelo, suav, imagenes, estado, mostrar_lm)
        if estado.get("caja") is None:
            texto(frame, "Ponte delante de la camara", 10, frame.shape[0] - 40, 0.7)

        ahora = time.time()
        t_ant = ahora
        ms = f"{res[2]:.1f} ms" if res else "-"
        texto(frame, f"{modelo.nombre} [{modo % len(MODOS) + 1}/{len(MODOS)}] | {ms} ", 10, 25, 0.6, (0, 255, 255))
        texto(frame, "n/b partido  m modelo  l landmarks  f pantalla  Esc salir",
              10, frame.shape[0] - 12, 0.45)
        selector.dibujar(frame)
        cv2.imshow(VENTANA, frame)

        k = cv2.waitKey(1) & 0xFF
        if k == 27:
            break
        elif k in (ord("n"), ord("b")):
            selector.siguiente() if k == ord("n") else selector.anterior()
            imagenes = reacciones_partido()
        elif k == ord("m"):
            for _ in MODOS:                       # salta los modelos que no se puedan cargar
                modo += 1
                try:
                    modelo = obtener_modelo(modo)
                    break
                except Exception as e:
                    print("No disponible:", e)
            suav.reset()
        elif k == ord("l"):
            mostrar_lm = not mostrar_lm
        elif k == ord("f"):
            full = not full
            cv2.setWindowProperty(VENTANA, cv2.WND_PROP_FULLSCREEN,
                                  cv2.WINDOW_FULLSCREEN if full else cv2.WINDOW_NORMAL)
finally:
    cam.release()
    cv2.destroyAllWindows()